In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.makedirs("/tmp/mplconfig", exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RANDOM_STATE = 42
LIKERT_MIN = 1
LIKERT_MAX = 5
EPS = 1e-8

ITEM_GROUPS = {
    "E": [f"E{i}" for i in range(1, 11)],
    "N": [f"N{i}" for i in range(1, 11)],
    "A": [f"A{i}" for i in range(1, 11)],
    "C": [f"C{i}" for i in range(1, 11)],
    "O": [f"O{i}" for i in range(1, 11)],
}

REVERSE_ITEMS = {
    "E2", "E4", "E6", "E8", "E10",
    "N2", "N4", "N6", "N8", "N10",
    "A1", "A3", "A5", "A7", "A9",
    "C2", "C4", "C6", "C8", "C10",
    "O2", "O4", "O6", "O8", "O10",
}


def standardize(X: np.ndarray):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0, ddof=0)
    sigma = np.where(sigma < EPS, 1.0, sigma)
    return (X - mu) / sigma, mu, sigma


def pca_2d(X: np.ndarray):
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    pcs = Xc @ Vt[:2].T
    var = (S ** 2) / max(1, (len(X) - 1))
    expl = var / var.sum()
    return pcs, expl[:2]


def squared_dist(A: np.ndarray, B: np.ndarray):
    return ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)


def kmeans_fit_predict(X: np.ndarray, k: int, seed=RANDOM_STATE, n_init=10, max_iter=120):
    rng = np.random.default_rng(seed)
    best_labels = None
    best_centers = None
    best_inertia = np.inf

    n = len(X)
    for _ in range(n_init):
        centers = X[rng.choice(n, size=k, replace=False)].copy()
        labels = np.zeros(n, dtype=int)
        for _ in range(max_iter):
            d2 = squared_dist(X, centers)
            new_labels = d2.argmin(axis=1)
            if np.array_equal(new_labels, labels):
                break
            labels = new_labels
            for c in range(k):
                mask = labels == c
                if mask.any():
                    centers[c] = X[mask].mean(axis=0)
                else:
                    centers[c] = X[rng.integers(0, n)]
        inertia = ((X - centers[labels]) ** 2).sum()
        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels.copy()
            best_centers = centers.copy()

    return best_labels, best_centers, float(best_inertia)


def _log_gauss_diag(X, mean, var):
    d = X.shape[1]
    var = np.maximum(var, 1e-6)
    diff = X - mean
    return -0.5 * (d * np.log(2 * np.pi) + np.log(var).sum() + ((diff ** 2) / var).sum(axis=1))


def _log_gauss_full(X, mean, cov):
    d = X.shape[1]
    cov = cov + np.eye(d) * 1e-6
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        cov = cov + np.eye(d) * 1e-3
        sign, logdet = np.linalg.slogdet(cov)
    inv = np.linalg.inv(cov)
    diff = X - mean
    quad = np.einsum("ni,ij,nj->n", diff, inv, diff)
    return -0.5 * (d * np.log(2 * np.pi) + logdet + quad)


def gmm_fit_predict(X: np.ndarray, k: int, cov_type="full", seed=RANDOM_STATE, n_init=3, max_iter=80, tol=1e-4):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    best_ll = -np.inf
    best_labels = None

    for _ in range(n_init):
        means = X[rng.choice(n, size=k, replace=False)].copy()
        weights = np.full(k, 1.0 / k)
        if cov_type == "diag":
            covs = np.tile(X.var(axis=0) + 1e-2, (k, 1))
        else:
            base_cov = np.cov(X, rowvar=False) + np.eye(d) * 1e-2
            covs = np.stack([base_cov.copy() for _ in range(k)], axis=0)

        prev_ll = None
        for _ in range(max_iter):
            log_prob = np.zeros((n, k))
            for j in range(k):
                if cov_type == "diag":
                    log_prob[:, j] = np.log(weights[j] + EPS) + _log_gauss_diag(X, means[j], covs[j])
                else:
                    log_prob[:, j] = np.log(weights[j] + EPS) + _log_gauss_full(X, means[j], covs[j])

            max_log = log_prob.max(axis=1, keepdims=True)
            resp = np.exp(log_prob - max_log)
            denom = resp.sum(axis=1, keepdims=True) + EPS
            resp /= denom
            ll = float((max_log + np.log(denom)).sum())

            Nk = resp.sum(axis=0) + EPS
            weights = Nk / n
            means = (resp.T @ X) / Nk[:, None]

            if cov_type == "diag":
                for j in range(k):
                    diff = X - means[j]
                    covs[j] = (resp[:, j][:, None] * (diff ** 2)).sum(axis=0) / Nk[j] + 1e-6
            else:
                for j in range(k):
                    diff = X - means[j]
                    cov = (resp[:, j][:, None] * diff).T @ diff / Nk[j]
                    covs[j] = cov + np.eye(d) * 1e-6

            if prev_ll is not None and abs(ll - prev_ll) < tol:
                break
            prev_ll = ll

        labels = resp.argmax(axis=1)
        if ll > best_ll:
            best_ll = ll
            best_labels = labels.copy()

    return best_labels, float(best_ll)


def adjusted_rand_index(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    n = y_true.size
    if n == 0:
        return np.nan

    classes, class_idx = np.unique(y_true, return_inverse=True)
    clusters, cluster_idx = np.unique(y_pred, return_inverse=True)
    contingency = np.zeros((classes.size, clusters.size), dtype=np.int64)
    np.add.at(contingency, (class_idx, cluster_idx), 1)

    def comb2(x):
        return x * (x - 1) / 2

    sum_comb_c = comb2(contingency).sum()
    sum_comb_rows = comb2(contingency.sum(axis=1)).sum()
    sum_comb_cols = comb2(contingency.sum(axis=0)).sum()
    total_comb = comb2(n)
    expected = (sum_comb_rows * sum_comb_cols) / (total_comb + EPS)
    max_index = 0.5 * (sum_comb_rows + sum_comb_cols)
    return float((sum_comb_c - expected) / (max_index - expected + EPS))


def silhouette_score_sample(X: np.ndarray, labels: np.ndarray, sample_size=3500, seed=RANDOM_STATE):
    n = len(X)
    if n <= 2:
        return np.nan
    rng = np.random.default_rng(seed)
    if n > sample_size:
        idx = rng.choice(n, size=sample_size, replace=False)
        Xs = X[idx]
        ls = labels[idx]
    else:
        Xs = X
        ls = labels

    D = np.sqrt(squared_dist(Xs, Xs))
    unique = np.unique(ls)
    sil = []
    for i in range(len(Xs)):
        same = ls == ls[i]
        if same.sum() <= 1:
            sil.append(0.0)
            continue
        a = D[i, same].sum() / (same.sum() - 1)
        b = np.inf
        for c in unique:
            if c == ls[i]:
                continue
            other = ls == c
            if other.sum() == 0:
                continue
            b = min(b, D[i, other].mean())
        sil.append((b - a) / max(a, b, EPS))
    return float(np.mean(sil))


def davies_bouldin_score_np(X: np.ndarray, labels: np.ndarray):
    unique = np.unique(labels)
    k = len(unique)
    if k < 2:
        return np.nan
    centroids = []
    scatters = []
    for c in unique:
        pts = X[labels == c]
        cent = pts.mean(axis=0)
        centroids.append(cent)
        scatters.append(np.sqrt(((pts - cent) ** 2).sum(axis=1)).mean())
    centroids = np.array(centroids)
    scatters = np.array(scatters)

    M = np.sqrt(squared_dist(centroids, centroids)) + EPS
    R = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if i == j:
                continue
            R[i, j] = (scatters[i] + scatters[j]) / M[i, j]
    return float(np.mean(R.max(axis=1)))


def calinski_harabasz_score_np(X: np.ndarray, labels: np.ndarray):
    n = len(X)
    unique = np.unique(labels)
    k = len(unique)
    if k < 2 or n <= k:
        return np.nan

    overall = X.mean(axis=0)
    B = 0.0
    W = 0.0
    for c in unique:
        pts = X[labels == c]
        nc = len(pts)
        cent = pts.mean(axis=0)
        B += nc * ((cent - overall) ** 2).sum()
        W += ((pts - cent) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k) + EPS))


def eval_internal(X: np.ndarray, labels: np.ndarray):
    return {
        "silhouette": silhouette_score_sample(X, labels),
        "davies_bouldin": davies_bouldin_score_np(X, labels),
        "calinski_harabasz": calinski_harabasz_score_np(X, labels),
    }


def choose_best(df: pd.DataFrame):
    rank = df.copy()
    rank["sil_rank"] = rank["silhouette"].rank(ascending=False, method="min")
    rank["db_rank"] = rank["davies_bouldin"].rank(ascending=True, method="min")
    rank["ch_rank"] = rank["calinski_harabasz"].rank(ascending=False, method="min")
    rank["rank_sum"] = rank[["sil_rank", "db_rank", "ch_rank"]].sum(axis=1)
    return rank.sort_values(["rank_sum", "silhouette"], ascending=[True, False]).iloc[0]


def reverse_code(series: pd.Series):
    return (LIKERT_MIN + LIKERT_MAX) - series


def response_qc_flags(df_raw: pd.DataFrame, item_cols: list[str]):
    items = df_raw[item_cols].to_numpy()

    def max_prop(row):
        _, counts = np.unique(row, return_counts=True)
        return counts.max() / counts.sum()

    straight_prop = np.apply_along_axis(max_prop, 1, items)
    straight_line = straight_prop >= 0.90

    pair_diffs = []
    for trait in ["E", "N", "A", "C", "O"]:
        for i in range(1, 10, 2):
            c1 = f"{trait}{i}"
            c2 = f"{trait}{i+1}"
            if c1 in REVERSE_ITEMS:
                diff = (reverse_code(df_raw[c1]) - df_raw[c2]).abs()
            elif c2 in REVERSE_ITEMS:
                diff = (df_raw[c1] - reverse_code(df_raw[c2])).abs()
            else:
                diff = (df_raw[c1] - df_raw[c2]).abs()
            pair_diffs.append(diff)

    inconsistency_score = pd.concat(pair_diffs, axis=1).mean(axis=1)
    reverse_inconsistent = inconsistency_score > 2.2

    flags = pd.DataFrame({
        "straight_line": straight_line,
        "reverse_inconsistent": reverse_inconsistent,
        "straight_prop": straight_prop,
        "reverse_inconsistency_score": inconsistency_score,
    })
    flags["low_quality"] = flags["straight_line"] | flags["reverse_inconsistent"]
    return flags


def engineer_features(df_raw: pd.DataFrame):
    df = df_raw.copy()
    for c in REVERSE_ITEMS:
        df[c] = reverse_code(df[c])

    for trait, cols in ITEM_GROUPS.items():
        df[f"{trait}_score"] = df[cols].mean(axis=1)

    trait_cols = ["O_score", "C_score", "E_score", "A_score", "N_score"]
    z, _, _ = standardize(df[trait_cols].to_numpy())
    zdf = pd.DataFrame(z, columns=[f"z_{c}" for c in trait_cols], index=df.index)
    df = pd.concat([df, zdf], axis=1)

    df["N_proxy"] = 0.7 * df["z_O_score"] + 0.3 * df["z_E_score"]
    df["J_proxy"] = 0.8 * df["z_C_score"] + 0.2 * df["z_A_score"]
    df["T_proxy"] = 0.6 * df["z_C_score"] - 0.4 * df["z_A_score"]

    df["Impulsivity_Index"] = -df["z_C_score"] + df["z_N_score"]
    df["Emotional_Reactivity"] = df["z_N_score"] - 0.3 * df["z_A_score"]
    df["Creative_Divergence"] = df["z_O_score"] - 0.2 * df["z_C_score"]
    df["Social_Adaptability"] = df["z_E_score"] + 0.5 * df["z_A_score"] - 0.3 * df["z_N_score"]
    return df


def run_kmeans_grid(X: np.ndarray, k_list):
    rows = []
    for k in k_list:
        labels, _, _ = kmeans_fit_predict(X, k, seed=RANDOM_STATE, n_init=12)
        rows.append({"method": "kmeans", "k": k, **eval_internal(X, labels)})
    return pd.DataFrame(rows)


def run_gmm_grid(X: np.ndarray, k_list, cov_types=("full", "diag")):
    rows = []
    for k in k_list:
        for cov in cov_types:
            labels, _ = gmm_fit_predict(X, k, cov_type=cov, seed=RANDOM_STATE, n_init=3)
            rows.append({"method": f"gmm_{cov}", "k": k, **eval_internal(X, labels)})
    return pd.DataFrame(rows)


def seed_stability(X: np.ndarray, fit_fn, seeds=range(10, 20)):
    label_list = [fit_fn(X, s) for s in seeds]
    aris = []
    for i in range(len(label_list)):
        for j in range(i + 1, len(label_list)):
            aris.append(adjusted_rand_index(label_list[i], label_list[j]))
    return float(np.mean(aris)) if aris else np.nan


def subsample_stability(X: np.ndarray, fit_fn, random_state=RANDOM_STATE, n_iter=12, frac=0.8):
    rng = np.random.default_rng(random_state)
    n = len(X)
    m = int(n * frac)
    samples = []
    for i in range(n_iter):
        idx = np.sort(rng.choice(n, size=m, replace=False))
        labels = fit_fn(X[idx], random_state + i)
        samples.append((idx, labels))

    aris = []
    for i in range(len(samples) - 1):
        idx_i, lab_i = samples[i]
        idx_j, lab_j = samples[i + 1]
        common, ii, jj = np.intersect1d(idx_i, idx_j, return_indices=True)
        if len(common) < 200:
            continue
        aris.append(adjusted_rand_index(lab_i[ii], lab_j[jj]))
    return float(np.mean(aris)) if aris else np.nan


def save_scatter_pca(X, labels, title, outpath, random_state=RANDOM_STATE):
    """2D PCA scatter colored by cluster labels."""
    pcs, expl = pca_2d(X)
    plt.figure(figsize=(7, 5))
    uniq = np.unique(labels)

    # Use default matplotlib cycle colors
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    for j, u in enumerate(uniq):
        mask = (labels == u)
        c = colors[j % len(colors)] if colors else None
        plt.scatter(pcs[mask, 0], pcs[mask, 1], s=16, alpha=0.6, label=f"Cluster {u}", color=c)

    plt.xlabel(f"PC1 ({expl[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({expl[1]*100:.1f}%)")
    plt.title(title)
    plt.legend(frameon=True, fontsize=9, loc="best")
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()


def save_profile_bar(df, cluster_col, cols, title, out_path):
    # df: feature dataframe
    # cluster_col: column name for cluster labels
    # cols: features to plot
    # title: plot title
    # out_path: where to save figure
    
    # compute cluster means
    means = df.groupby(cluster_col)[cols].mean().sort_index()

    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4.8))
    means.T.plot(kind="bar")  # each cluster as a bar group
    plt.title(title)
    plt.xlabel("Feature")
    plt.ylabel("Mean (z-scored / scaled)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_heatmap(df: pd.DataFrame, label_col: str, feat_cols: list[str], title: str, outpath: Path):
    """Save a centroid heatmap using matplotlib (seaborn-free)."""
    cent = df.groupby(label_col)[feat_cols].mean()
    plt.figure(figsize=(10, 4.8))

    data = cent.values
    vmax = float(np.nanmax(np.abs(data))) if data.size else 1.0
    im = plt.imshow(data, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, label="Mean")

    plt.xticks(ticks=np.arange(len(feat_cols)), labels=feat_cols, rotation=35, ha="right")
    plt.yticks(ticks=np.arange(len(cent.index)), labels=[str(i) for i in cent.index])

    # annotate cells
    for r in range(data.shape[0]):
        for c in range(data.shape[1]):
            plt.text(c, r, f"{data[r, c]:.2f}", ha="center", va="center", fontsize=8)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


def main():
    # seaborn not available; using matplotlib styling only

    data_path = Path("PartB_data.csv")
    out_dir = Path("outputs/partB")
    fig_dir = out_dir / "figures"
    out_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    df_raw = pd.read_csv(data_path, sep="\t")
    item_cols = [c for cols in ITEM_GROUPS.values() for c in cols]
    df_raw = df_raw.dropna(subset=item_cols).copy()

    qc = response_qc_flags(df_raw, item_cols)
    df_qc = pd.concat([df_raw.reset_index(drop=True), qc], axis=1)
    df_clean = df_qc.loc[~df_qc["low_quality"]].copy()

    df_feat = engineer_features(df_clean)
    big5_cols = ["O_score", "C_score", "E_score", "A_score", "N_score"]
    eng_cols = ["N_proxy", "J_proxy", "T_proxy", "Impulsivity_Index", "Emotional_Reactivity", "Creative_Divergence", "Social_Adaptability"]

    X_big5, _, _ = standardize(df_feat[big5_cols].to_numpy())
    X_full, _, _ = standardize(df_feat[big5_cols + eng_cols].to_numpy())

    k_list = list(range(2, 9))
    res_big5 = pd.concat([run_kmeans_grid(X_big5, k_list), run_gmm_grid(X_big5, k_list)], ignore_index=True)
    res_full = pd.concat([run_kmeans_grid(X_full, k_list), run_gmm_grid(X_full, k_list)], ignore_index=True)
    res_big5.insert(0, "representation", "big5_only")
    res_full.insert(0, "representation", "big5_plus_engineered")
    results = pd.concat([res_big5, res_full], ignore_index=True)
    results.to_csv(out_dir / "internal_metrics.csv", index=False)

    best_big5 = choose_best(res_big5)
    best_full = choose_best(res_full)

    # Selection priority for report consistency:
    # 1) Primary narrative split: best K-Means within engineered representation.
    # 2) Cross-method best: secondary fine-grained validation across K-Means + GMM.
    kmeans_best = choose_best(res_full[res_full["method"] == "kmeans"])
    km_k = int(kmeans_best["k"])
    km_labels, _, _ = kmeans_fit_predict(X_full, km_k, seed=RANDOM_STATE, n_init=15)
    df_feat["cluster_kmeans"] = km_labels

    gmm_best = choose_best(res_full[res_full["method"].str.startswith("gmm_")])
    gmm_k = int(gmm_best["k"])
    gmm_cov = gmm_best["method"].split("_")[1]
    gmm_labels, _ = gmm_fit_predict(X_full, gmm_k, cov_type=gmm_cov, seed=RANDOM_STATE, n_init=4)
    df_feat["cluster_gmm"] = gmm_labels

    cross_best = best_full
    if cross_best["method"] == "kmeans":
        cross_k = int(cross_best["k"])
        cross_labels, _, _ = kmeans_fit_predict(X_full, cross_k, seed=RANDOM_STATE, n_init=15)
    else:
        cross_k = int(cross_best["k"])
        cross_cov = cross_best["method"].split("_")[1]
        cross_labels, _ = gmm_fit_predict(X_full, cross_k, cov_type=cross_cov, seed=RANDOM_STATE, n_init=4)
    df_feat["cluster_best"] = cross_labels

    def fit_seed_primary(X, s):
        return kmeans_fit_predict(X, km_k, seed=s, n_init=8)[0]

    seed_ari = seed_stability(X_full, fit_seed_primary)
    subsample_ari = subsample_stability(X_full, fit_seed_primary)

    selection_summary = pd.DataFrame([
        {
            "selection_role": "primary_narrative",
            "representation": "big5_plus_engineered",
            "method": "kmeans",
            "k": km_k,
            "cov_type": "",
            "criterion": "kmeans_family_priority_then_separability",
            "silhouette": float(kmeans_best["silhouette"]),
            "davies_bouldin": float(kmeans_best["davies_bouldin"]),
            "calinski_harabasz": float(kmeans_best["calinski_harabasz"]),
            "rank_sum": float(kmeans_best["rank_sum"]),
        },
        {
            "selection_role": "cross_method_alternative",
            "representation": "big5_plus_engineered",
            "method": str(cross_best["method"]),
            "k": int(cross_best["k"]),
            "cov_type": str(cross_best["method"]).split("_")[1] if str(cross_best["method"]).startswith("gmm_") else "",
            "criterion": "composite_rank_sum_across_methods",
            "silhouette": float(cross_best["silhouette"]),
            "davies_bouldin": float(cross_best["davies_bouldin"]),
            "calinski_harabasz": float(cross_best["calinski_harabasz"]),
            "rank_sum": float(cross_best["rank_sum"]),
        },
        {
            "selection_role": "gmm_family_reference",
            "representation": "big5_plus_engineered",
            "method": str(gmm_best["method"]),
            "k": int(gmm_best["k"]),
            "cov_type": str(gmm_best["method"]).split("_")[1],
            "criterion": "gmm_family_rank_sum",
            "silhouette": float(gmm_best["silhouette"]),
            "davies_bouldin": float(gmm_best["davies_bouldin"]),
            "calinski_harabasz": float(gmm_best["calinski_harabasz"]),
            "rank_sum": float(gmm_best["rank_sum"]),
        },
    ])
    selection_summary.to_csv(out_dir / "selection_summary.csv", index=False)

    stability_summary = pd.DataFrame([
        {
            "selection_role": "primary_narrative",
            "method": "kmeans",
            "k": km_k,
            "seed_ari": float(seed_ari),
            "subsample_ari": float(subsample_ari),
        }
    ])
    stability_summary.to_csv(out_dir / "stability_summary.csv", index=False)

    sens_rows = []
    base_labels = km_labels
    for a, b in [(0.7, 0.3), (1.0, 0.0), (0.5, 0.5)]:
        n_proxy = a * df_feat["z_O_score"] + b * df_feat["z_E_score"]
        alt = df_feat[big5_cols + ["J_proxy", "T_proxy", "Impulsivity_Index", "Emotional_Reactivity", "Creative_Divergence", "Social_Adaptability"]].copy()
        alt["N_proxy"] = n_proxy
        X_alt, _, _ = standardize(alt.to_numpy())
        alt_labels, _, _ = kmeans_fit_predict(X_alt, km_k, seed=RANDOM_STATE, n_init=12)
        sens_rows.append({
            "N_proxy": f"{a:.1f}*O + {b:.1f}*E",
            "corr_with_O": float(np.corrcoef(n_proxy, df_feat["z_O_score"])[0, 1]),
            "corr_with_E": float(np.corrcoef(n_proxy, df_feat["z_E_score"])[0, 1]),
            "ARI_vs_baseline_kmeans": adjusted_rand_index(base_labels, alt_labels),
            **eval_internal(X_alt, alt_labels),
        })
    sens_df = pd.DataFrame(sens_rows)
    sens_df.to_csv(out_dir / "sensitivity_n_proxy.csv", index=False)

    proxy_cols = ["N_proxy", "J_proxy", "T_proxy", "Impulsivity_Index"]
    src_cols = ["z_O_score", "z_C_score", "z_E_score", "z_A_score", "z_N_score"]
    corr = df_feat[proxy_cols + src_cols].corr().loc[proxy_cols, src_cols]
    corr.to_csv(out_dir / "proxy_correlation.csv")

    vis_cols = ["O_score", "C_score", "E_score", "A_score", "N_score", "N_proxy", "J_proxy", "T_proxy", "Impulsivity_Index"]
    save_scatter_pca(X_full, km_labels, f"KMeans PCA (k={km_k})", fig_dir / "kmeans_pca_scatter.png")
    save_profile_bar(df_feat, "cluster_kmeans", vis_cols, "KMeans Cluster Profiles", fig_dir / "kmeans_cluster_profile_bar.png")
    save_heatmap(df_feat, "cluster_kmeans", vis_cols, "KMeans Centroid Heatmap", fig_dir / "kmeans_centroid_heatmap.png")

    save_scatter_pca(X_full, gmm_labels, f"GMM PCA (k={gmm_k}, cov={gmm_cov})", fig_dir / "gmm_pca_scatter.png")
    save_profile_bar(df_feat, "cluster_gmm", vis_cols, "GMM Cluster Profiles", fig_dir / "gmm_cluster_profile_bar.png")
    save_heatmap(df_feat, "cluster_gmm", vis_cols, "GMM Centroid Heatmap", fig_dir / "gmm_centroid_heatmap.png")

    top = results.sort_values("silhouette", ascending=False).head(15).copy()
    top["label"] = top["representation"] + "|" + top["method"] + "|k=" + top["k"].astype(str)

    plt.figure(figsize=(12, 5.5))
    top_sorted = top.sort_values("silhouette", ascending=False).reset_index(drop=True)
    
    x = np.arange(len(top_sorted))
    plt.bar(x, top_sorted["silhouette"].values)
    
    plt.xticks(x, top_sorted["label"].values, rotation=60, ha="right")
    plt.ylabel("Silhouette")
    plt.title("Top Silhouette Configurations")
    
    plt.tight_layout()
    plt.savefig(fig_dir / "top_silhouette_configs.png", dpi=200)
    plt.close()

    export_cols = ["country", "age", "gender"] + big5_cols + eng_cols + ["cluster_kmeans", "cluster_gmm", "cluster_best"]
    for c in export_cols:
        if c not in df_feat.columns:
            df_feat[c] = np.nan
    df_feat[export_cols].to_csv(out_dir / "cluster_assignments.csv", index=False)

    df_feat.groupby("cluster_kmeans")[big5_cols + eng_cols].mean().to_csv(out_dir / "kmeans_cluster_profile.csv")
    df_feat.groupby("cluster_gmm")[big5_cols + eng_cols].mean().to_csv(out_dir / "gmm_cluster_profile.csv")

    print("Selection priority: primary_narrative uses K-Means family; cross_method_alternative is reported as fine-grained validation.")
    print(selection_summary.to_string(index=False))
    print(f"Primary stability (KMeans k={km_k}): seed_ari={seed_ari:.3f}, subsample_ari={subsample_ari:.3f}")
    print(f"Figures written to: {fig_dir}")


if __name__ == "__main__":
    main()

Report written to: outputs/partB/partB_report.md
Figures written to: outputs/partB/figures
